# Picoboi: fast chat corpus pilot
Run in order with **Runtime > Change runtime type > GPU**. A free GPU is subject to availability and usage limits. No existing API endpoint is needed: the teacher runs inside this notebook. This notebook has been statically checked, but has not yet been executed on a Colab GPU.

Start with the pilot. Inspect its outputs before running the optional 10M-word cell. The vocabulary has 968 words. Existing project data stays intact. Teacher-generated examples remain separate from directly authored data.


In [ ]:
import subprocess, sys
from pathlib import Path
subprocess.run(['nvidia-smi'], check=True)
repo = Path('/content/Picobot')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/farzeenilyaszargar/Picobot.git', str(repo)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'vllm'], check=True)


Save output to your Google Drive so it survives a runtime reset. Google will ask you to authorize the mount. Each flush has storage overhead, which will be reflected in the measured generation rate. Existing output can resume when the recorded configuration matches.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
output = Path('/content/drive/MyDrive/Picoboi/fast-chat-v1')
output.mkdir(parents=True, exist_ok=True)
with (output / 'environment.txt').open('w') as f:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=f, check=True)


In [ ]:
import time, urllib.request, urllib.error
MODEL = 'Qwen/Qwen2.5-7B-Instruct-AWQ'
log = Path('/content/picoboi-teacher.log').open('a')
server = subprocess.Popen([
    'vllm', 'serve', MODEL, '--host', '127.0.0.1', '--port', '8000',
    '--dtype', 'half', '--max-model-len', '6144', '--max-num-seqs', '16',
    '--enable-prefix-caching'], stdout=log, stderr=subprocess.STDOUT)
for _ in range(180):
    if server.poll() is not None:
        raise RuntimeError(Path('/content/picoboi-teacher.log').read_text()[-8000:])
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=5) as response:
            if response.status == 200:
                print('Teacher ready')
                break
    except (urllib.error.URLError, TimeoutError):
        time.sleep(5)
else:
    server.terminate()
    raise RuntimeError('Teacher not ready. Inspect /content/picoboi-teacher.log before retrying.')


Run the pilot first. The runner checks vocabulary, roles, context limits under both tokenizers, and duplicates. It does not automatically judge whether the answers are sensible.


In [ ]:
command = [sys.executable, str(repo / 'training/fast_corpus/generate.py'),
           '--model', MODEL, '--base-url', 'http://127.0.0.1:8000/v1',
           '--output', str(output), '--workers', '8']
subprocess.run(command + ['--max-jobs', '100', '--target-words', '10000'], cwd=repo, check=True)


In [ ]:
import json
print((output / 'report.json').read_text())
with (output / 'accepted.jsonl').open() as f:
    for i,line in enumerate(f):
        if i >= 10: break
        row = json.loads(line)
        print('\n', row['id'], row['profile'])
        print(row['messages_text'])
print('Review more examples before proceeding. A mechanical pass is not a semantic quality pass.')


Optional production run. Change the flag only after inspecting the pilot. This can exceed a free session's limits. It targets 10M English words; token counts will be larger. It can resume on a later session if Drive output and the recorded configuration match. The current run budget is 100,000 new requests, not a cost limit. Use a rented GPU if guaranteed sustained runtime matters.


In [ ]:
RUN_FULL_CORPUS = False
if RUN_FULL_CORPUS:
    subprocess.run(command + ['--max-jobs', '100000', '--target-words', '10000000'], cwd=repo, check=True)
else:
    print('Full run disabled. Review the pilot first.')


Stop the teacher when finished. Outputs remain in Drive. On a rented server, terminating this process does not stop instance billing: stop the cloud instance separately.


In [ ]:
server.terminate()
try:
    server.wait(timeout=30)
except subprocess.TimeoutExpired:
    server.kill()
log.close()
